In [1]:
#promts to put in the anaconda prompt 1. conda activate pyspark_env 2. jupyter notebook

import sys
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as s
from pyspark.sql.types import DoubleType, StringType

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = SparkSession.builder \
    .appName("Testing PySpark Example") \
    .master("local[*]") \
    .config("spark.sql.parquet.enableVectorizedReader", "false") \
    .config("spark.driver.memory", "4g")\
    .getOrCreate()

In [2]:
df_raw = spark.read.parquet('2014-2017/')


#Check data types
df_raw.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: integer (nullable = true)



In [3]:
#change numerical datatypes to doubles
df = df_raw

for col_name, data_type in df_raw.dtypes:
    if data_type in ['bigint', 'int']:
        df = df.withColumn(col_name, s.col(col_name).cast(DoubleType()))
            
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|     2.0| 2014-01-01 00:02:00|  2014-01-01 00:04:00|            6.0|          0.0|       1.0|              NULL|       146.0|       146.0|         1.0|        3.5|  0.5|    0.5|      0.0

In [4]:
df.printSchema()

root
 |-- VendorID: double (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: double (nullable = true)
 |-- DOLocationID: double (nullable = true)
 |-- payment_type: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [5]:
entries = df.count()
print(entries)

521176000


In [6]:
#check for nulls 

nulls = df.select([
    s.count(s.when(s.col(c).isNull(), 1)).alias(f"{c}_nulls")
    for c in df.columns
]).toPandas()

print(nulls)


   VendorID_nulls  tpep_pickup_datetime_nulls  tpep_dropoff_datetime_nulls  \
0               0                           0                            0   

   passenger_count_nulls  trip_distance_nulls  RatecodeID_nulls  \
0                      0                    0                 0   

   store_and_fwd_flag_nulls  PULocationID_nulls  DOLocationID_nulls  \
0                  50120238                   0                   0   

   payment_type_nulls  fare_amount_nulls  extra_nulls  mta_tax_nulls  \
0                   0                  0            0              0   

   tip_amount_nulls  tolls_amount_nulls  improvement_surcharge_nulls  \
0                 0                   0                     53750503   

   total_amount_nulls  congestion_surcharge_nulls  airport_fee_nulls  
0                   0                   521175998          521176000  


In [7]:
print(type(nulls))

<class 'pandas.core.frame.DataFrame'>


In [8]:
#check for NaNs
nans = df.select([
    s.count(s.when(s.isnan(s.col(c)), 1)).alias(f"{c}_nans")
    for c, data_type in df.dtypes
    if data_type in ['double', 'float']
]).toPandas()

print(nans)

   VendorID_nans  passenger_count_nans  trip_distance_nans  RatecodeID_nans  \
0              0                     0                   0                0   

   PULocationID_nans  DOLocationID_nans  payment_type_nans  fare_amount_nans  \
0                  0                  0                  0                 0   

   extra_nans  mta_tax_nans  tip_amount_nans  tolls_amount_nans  \
0           0             0                0                  0   

   improvement_surcharge_nans  total_amount_nans  congestion_surcharge_nans  \
0                           0                  0                          0   

   airport_fee_nans  
0                 0  


In [9]:
cleaned_df = df.dropDuplicates()

In [10]:
#remove the columns store_and_fwd_flag, fare_amount, extra, mta_tax, tolls_amount, improvement_surcharge, congestion_surcharge

cleaned_df = df.drop("store_and_fwd_flag", "extra", "mta_tax", "tolls_amount", "improvement_surcharge", "congestion_surcharge", "airport_fee")

cleaned_df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------+------------+------------+-----------+----------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|PULocationID|DOLocationID|payment_type|fare_amount|tip_amount|total_amount|
+--------+--------------------+---------------------+---------------+-------------+----------+------------+------------+------------+-----------+----------+------------+
|     2.0| 2014-01-01 00:02:00|  2014-01-01 00:04:00|            6.0|          0.0|       1.0|       146.0|       146.0|         1.0|        3.5|      0.02|        4.52|
|     2.0| 2014-01-01 00:06:00|  2014-01-01 00:09:00|            5.0|          0.0|       1.0|       146.0|       146.0|         1.0|        3.5|      0.05|        4.55|
|     2.0| 2014-01-01 00:10:00|  2014-01-01 00:13:00|            5.0|          0.0|       1.0|       146.0|       146.0|         1.0|        3.5|     

In [11]:
# adding the additional columns called trip_duration and average_trip_speed
# calculate duration by subtracting pickup and dropoff

cleaned_df = cleaned_df.withColumn("trip_duration_interval", s.col("tpep_dropoff_datetime")-s.col("tpep_pickup_datetime"))
cleaned_df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------+------------+------------+-----------+----------+------------+----------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|PULocationID|DOLocationID|payment_type|fare_amount|tip_amount|total_amount|trip_duration_interval|
+--------+--------------------+---------------------+---------------+-------------+----------+------------+------------+------------+-----------+----------+------------+----------------------+
|     2.0| 2014-01-01 00:02:00|  2014-01-01 00:04:00|            6.0|          0.0|       1.0|       146.0|       146.0|         1.0|        3.5|      0.02|        4.52|  INTERVAL '0 00:02...|
|     2.0| 2014-01-01 00:06:00|  2014-01-01 00:09:00|            5.0|          0.0|       1.0|       146.0|       146.0|         1.0|        3.5|      0.05|        4.55|  INTERVAL '0 00:03...|
|     2.0| 2014-01-01 00:10:00|  20

In [12]:
#check the data type of the duration calculated

cleaned_df.printSchema()

root
 |-- VendorID: double (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- PULocationID: double (nullable = true)
 |-- DOLocationID: double (nullable = true)
 |-- payment_type: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- trip_duration_interval: interval day to second (nullable = true)



In [13]:
#convert the duration column into hours

cleaned_df = cleaned_df.withColumn(
    "trip_duration",
    s.expr("""
    extract(DAY from trip_duration_interval)*24 +
    extract(HOUR from trip_duration_interval)+
    extract(MINUTE from trip_duration_interval)/60+
    extract(SECOND from trip_duration_interval)/3600
    """)
)
cleaned_df = cleaned_df.drop("trip_duration_interval")

In [14]:
cleaned_df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------+------------+------------+-----------+----------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|PULocationID|DOLocationID|payment_type|fare_amount|tip_amount|total_amount|       trip_duration|
+--------+--------------------+---------------------+---------------+-------------+----------+------------+------------+------------+-----------+----------+------------+--------------------+
|     2.0| 2014-01-01 00:02:00|  2014-01-01 00:04:00|            6.0|          0.0|       1.0|       146.0|       146.0|         1.0|        3.5|      0.02|        4.52| 0.03333333333333333|
|     2.0| 2014-01-01 00:06:00|  2014-01-01 00:09:00|            5.0|          0.0|       1.0|       146.0|       146.0|         1.0|        3.5|      0.05|        4.55|                0.05|
|     2.0| 2014-01-01 00:10:00|  2014-01-01 0

In [15]:
#calculate the average speed in miles per hour
cleaned_df = cleaned_df.withColumn("average_trip_speed", s.col("trip_distance")/s.col("trip_duration"))

cleaned_df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------+------------+------------+-----------+----------+------------+--------------------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|PULocationID|DOLocationID|payment_type|fare_amount|tip_amount|total_amount|       trip_duration|average_trip_speed|
+--------+--------------------+---------------------+---------------+-------------+----------+------------+------------+------------+-----------+----------+------------+--------------------+------------------+
|     2.0| 2014-01-01 00:02:00|  2014-01-01 00:04:00|            6.0|          0.0|       1.0|       146.0|       146.0|         1.0|        3.5|      0.02|        4.52| 0.03333333333333333|               0.0|
|     2.0| 2014-01-01 00:06:00|  2014-01-01 00:09:00|            5.0|          0.0|       1.0|       146.0|       146.0|         1.0|        3.5|      0.05|    

In [16]:
cleaned_df.printSchema()

root
 |-- VendorID: double (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- PULocationID: double (nullable = true)
 |-- DOLocationID: double (nullable = true)
 |-- payment_type: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- trip_duration: double (nullable = true)
 |-- average_trip_speed: double (nullable = true)



In [19]:
#extraction of the min and max ranges from the raw dataset 

min_max_passengers = cleaned_df.select(s.min(s.col("passenger_count")), s.max(s.col("passenger_count")))
min_max_passengers.show()

trip_distance = cleaned_df.select(s.min(s.col("trip_distance")), s.max(s.col("trip_distance")))
trip_distance.show()

fare_amount = cleaned_df.select(s.min(s.col("fare_amount")), s.max(s.col("fare_amount")))
fare_amount.show()

trip_amount = cleaned_df.select(s.min(s.col("tip_amount")), s.max(s.col("tip_amount")))
trip_amount .show()

total_amount = cleaned_df.select(s.min(s.col("total_amount")), s.max(s.col("total_amount")))
total_amount.show()

trip_duration = cleaned_df.select(s.min(s.col("trip_duration")), s.max(s.col("trip_duration")))
trip_duration.show()

average_trip_speed = cleaned_df.select(s.min(s.col("average_trip_speed")), s.max(s.col("average_trip_speed")))
average_trip_speed.show()

+--------------------+--------------------+
|min(passenger_count)|max(passenger_count)|
+--------------------+--------------------+
|                 0.0|               208.0|
+--------------------+--------------------+

+------------------+------------------+
|min(trip_distance)|max(trip_distance)|
+------------------+------------------+
|     -4.08401244E7|     1.346190631E8|
+------------------+------------------+

+----------------+----------------+
|min(fare_amount)|max(fare_amount)|
+----------------+----------------+
|          -957.6|       861604.49|
+----------------+----------------+

+---------------+---------------+
|min(tip_amount)|max(tip_amount)|
+---------------+---------------+
|        -448.91|      3950588.8|
+---------------+---------------+

+-----------------+-----------------+
|min(total_amount)|max(total_amount)|
+-----------------+-----------------+
|           -958.4|      1.0000015E7|
+-----------------+-----------------+

+-------------------+--------------

In [20]:
#check the outlier ranges of some of these parameters to determine reasonable ways to filter them out

quantiles_fare = cleaned_df.approxQuantile("fare_amount", [0.25, 0.75], 0.01)
upper_bound_fare = quantiles_fare[1]+1.5*(quantiles_fare[1] - quantiles_fare[0])
print(upper_bound_fare)

quantiles_total = cleaned_df.approxQuantile("total_amount", [0.25, 0.75], 0.01)
upper_bound_total = quantiles_total[1]+1.5*(quantiles_total[1] - quantiles_total[0])
print(upper_bound_total)

quantiles_duration = cleaned_df.approxQuantile("trip_duration", [0.25, 0.75], 0.01)
upper_bound_duration = quantiles_duration[1]+1.5*(quantiles_duration[1] - quantiles_duration[0])
print(upper_bound_duration)

quantiles_speed = cleaned_df.approxQuantile("average_trip_speed", [0.25, 0.75], 0.01)
upper_bound_speed = quantiles_speed[1]+1.5*(quantiles_speed[1] - quantiles_speed[0])
print(upper_bound_speed)

27.75
31.250000000000004
0.5756944444533333
24.360913447433717


In [22]:
#filter for the correct years of the data set

cleaned_df = cleaned_df.withColumn("year",s.expr("""extract(YEAR from tpep_pickup_datetime)"""))
cleaned_df = cleaned_df.where((s.col("year") > 2013) & (s.col("year") < 2027))
df_entries = cleaned_df.count()
print(df_entries)
cleaned_df = cleaned_df.drop("year")

#trip distance > 0 and trip_distance 

cleaned_df = cleaned_df.where((s.col("trip_distance") > 0)& (s.col("trip_distance") < 20))
df_entries = cleaned_df.count()
print(df_entries)

#passenger_cout > 0  and passenger_count < 5
cleaned_df = cleaned_df.where((s.col("passenger_count") > 0)& (s.col("passenger_count") < 5))
df_entries = cleaned_df.count()
print(df_entries)


#trip duration > 0 hours and trip_duration < 3 hours

cleaned_df = cleaned_df.where((s.col("trip_duration") > 0) & (s.col("trip_duration") < 3))
df_entries = cleaned_df.count()
print(df_entries)

#average trip speed > 0 and less than the speed limit of 55 miles
cleaned_df = cleaned_df.where((s.col("average_trip_speed") > 0) & (s.col("average_trip_speed") < 55))
df_entries = cleaned_df.count()
print(df_entries)

#tip_amount > 0

cleaned_df = cleaned_df.where(s.col("tip_amount") >= 0)
entries_new = cleaned_df.count()
print(entries_new)

#total amount >0 and less than 2000 dollars

cleaned_df = cleaned_df.where((s.col("total_amount") > 0) & (s.col("total_amount") < 2000))
df_entries = cleaned_df.count()
print(df_entries)

#keep only the rows in which the fare amount is less than or equal to 100 dollars.

cleaned_df = cleaned_df.where((s.col("fare_amount") > 0)& (s.col("fare_amount") <= 100))
entries_new = cleaned_df.count()
print(entries_new)

521175885
515209320
470072219
469512290
468913829
468913068
468791050
468713819


In [23]:
cleaned_df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------+------------+------------+-----------+----------+------------+-------------------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|PULocationID|DOLocationID|payment_type|fare_amount|tip_amount|total_amount|      trip_duration|average_trip_speed|
+--------+--------------------+---------------------+---------------+-------------+----------+------------+------------+------------+-----------+----------+------------+-------------------+------------------+
|     1.0| 2014-01-01 00:29:18|  2014-01-01 00:35:13|            2.0|          1.8|       1.0|       229.0|       262.0|         2.0|        7.5|       0.0|         8.5|0.09861111111333333|18.253521126349217|
|     1.0| 2014-01-01 00:36:33|  2014-01-01 00:55:19|            3.0|          4.8|       1.0|       140.0|         7.0|         2.0|       17.5|       0.0|        

In [24]:
sorted_df = cleaned_df.sort("tpep_pickup_datetime")

In [25]:
sampled_df = sorted_df.sample(withReplacement = False, fraction = 0.2, seed = 42)
spark.conf.set("spark.sql.parquet.compression.codec.zstd.level", "9")
sampled_df.coalesce(4).write.mode("overwrite").option("compression", "zstd").parquet("C:/Users/savan/Documents/MIT STUDIES/5. MIT 805/GROUP PROJECT/YELLOW TAXI DATA/2014-2017_FINAL_CLEANED")

In [26]:
sampled_df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------+------------+------------+-----------+----------+------------+-------------------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|PULocationID|DOLocationID|payment_type|fare_amount|tip_amount|total_amount|      trip_duration|average_trip_speed|
+--------+--------------------+---------------------+---------------+-------------+----------+------------+------------+------------+-----------+----------+------------+-------------------+------------------+
|     2.0| 2014-01-01 00:00:00|  2014-01-01 00:10:00|            1.0|         3.65|       1.0|       114.0|        79.0|         2.0|       12.5|       0.0|        13.5|0.16666666666666666|21.900000000000002|
|     2.0| 2014-01-01 00:00:00|  2014-01-01 00:06:00|            2.0|         2.52|       1.0|       166.0|       142.0|         1.0|        8.5|       1.5|        

In [ ]:
entries_sampled = sampled_df.count()
print(entries_sampled)